#<font color='pink'> Training a model for image classification

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

from tensorflow import keras
from keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Activation, Dropout, Flatten, Dense
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from tensorflow.keras.preprocessing import image_dataset_from_directory

import os
import matplotlib.image as mpimg
from PIL import Image
from glob import glob

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, f1_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import zipfile

local_zip = './datasets_1000.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('./')
zip_ref.close()

In [ ]:
# WARNING: This command will delete all files and directories in the current working directory.
# Use with extreme caution!
# You will be prompted to confirm before deletion.
!rm -rf ./__MACOSX/
!rm -rf ./sample_data/
!rm datasets_1000.zip

In [ ]:
# =========================================
# ดูตัวอย่างภาพจริง
# =========================================
fig = plt.gcf()
fig.set_size_inches(9, 6)


cat_dir = './datasets_1000/train/cats'
dog_dir = './datasets_1000/train/dogs'

cat_names = os.listdir(cat_dir)
dog_names = os.listdir(dog_dir)

# Select first 3 images for each class
cat_images = [os.path.join(cat_dir, fname) for fname in cat_names[:3]]
dog_images = [os.path.join(dog_dir, fname) for fname in dog_names[:3]]

print(cat_images)

# Display 2 rows, 3 columns
for i, img_path in enumerate(cat_images + dog_images):
    sp = plt.subplot(2, 3, i+1)
    sp.axis('Off')

    img = mpimg.imread(img_path)
    plt.imshow(img)

plt.show()

#EDA

## Count Images per Class


> Add blockquote




ทำอะไร: นับจำนวนรูปในแต่ละคลาส แล้ว plot bar
ทำไม: เช็ค class imbalance


*   ถ้า imbalance มาก อาจต้องใช้ class_weight/oversampling
*   ถ้าใกล้เคียงกัน จะเทรนง่ายและ metric น่าเชื่อถือกว่า





In [ ]:
def count_images_per_class(train_dir):
    rows = []
    for cls in sorted(os.listdir(train_dir)):
        cls_path = os.path.join(train_dir, cls)
        if os.path.isdir(cls_path):
            n = len([f for f in os.listdir(cls_path) if f.lower().endswith((".jpg",".jpeg",".png"))])
            rows.append((cls, n))
    return pd.DataFrame(rows, columns=["class","count"]).sort_values("count", ascending=False)

TRAIN_DIR = './datasets_1000/train'
df_count = count_images_per_class(TRAIN_DIR)
print("Image counts per class:")
print(df_count)

plt.figure(figsize=(6,4))
plt.bar(df_count["class"], df_count["count"])
plt.title("Count Class")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

##  View Sample Images



ทำอะไร: สุ่มรูปแล้วเก็บ width/height ทำ scatter plot

ทำไม: ดูว่าขนาดภาพหลากหลายไหม ถ้าหลากหลายต้อง resize ให้เท่ากันก่อนเข้าโมเดล

In [ ]:
# กราฟกระจายขนาดภาพ
from PIL import Image

def get_image_dimensions(image_path):
    with Image.open(image_path) as img:
        return img.width, img.height

image_paths = []
for root, _, files in os.walk(TRAIN_DIR):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".gif")):
            image_paths.append(os.path.join(root, file))

dimensions = []
for path in image_paths:
    try:
        width, height = get_image_dimensions(path)
        dimensions.append({"width": width, "height": height})
    except Exception as e:
        print(f"Could not process image {path}: {e}")

df_size = pd.DataFrame(dimensions)

plt.figure(figsize=(6,5))
plt.scatter(df_size["width"], df_size["height"], s=10)
plt.title("Image size distribution (sample)")
plt.xlabel("Width")
plt.ylabel("Height")
plt.show()

In [ ]:
SAMPLE_N = 600
RANDOM_SEED = 42

def get_image_paths(train_dir):
    cats = glob(os.path.join(train_dir, "cats", "*"))
    dogs = glob(os.path.join(train_dir, "dogs", "*"))
    data = [(p, "cat") for p in cats] + [(p, "dog") for p in dogs]
    return data

def sample_image_paths(train_dir, n=SAMPLE_N, seed=RANDOM_SEED):
    data = get_image_paths(train_dir)
    rng = np.random.default_rng(seed)
    if n >= len(data):
        return data
    idx = rng.choice(len(data), size=n, replace=False)
    return [data[i] for i in idx]

def open_rgb(path):
    return Image.open(path).convert("RGB")

def pil_to_np(img):
    return np.array(img, dtype=np.float32)

def to_grayscale_np(rgb_np):
    return 0.299 * rgb_np[...,0] + 0.587 * rgb_np[...,1] + 0.114 * rgb_np[...,2]

print("SAMPLE_N, RANDOM_SEED, and helper functions defined.")

In [ ]:
def eda_aspect_ratio_and_size(train_dir, n=SAMPLE_N, seed=RANDOM_SEED):
    samples = sample_image_paths(train_dir, n=n, seed=seed)

    widths, heights, ratios = [], [], []

    for path, _ in samples:
        img = Image.open(path)
        w, h = img.size
        widths.append(w)
        heights.append(h)
        ratios.append(w / h)

    plt.figure()
    plt.hist(ratios, bins=30)
    plt.title("Aspect Ratio (w/h)")
    plt.show()

    plt.figure()
    plt.scatter(widths, heights, s=10)
    plt.title("Image Size Scatter")
    plt.show()

    print(f"Mean width: {np.mean(widths):.1f}")
    print(f"Mean height: {np.mean(heights):.1f}")
    print(f"Aspect ratio mean: {np.mean(ratios):.3f}")


In [ ]:
def eda_brightness_distribution(train_dir, n=SAMPLE_N, seed=RANDOM_SEED):
    samples = sample_image_paths(train_dir, n=n, seed=seed)

    means = []

    for path, _ in samples:
        img = open_rgb(path)
        arr = pil_to_np(img)
        gray = to_grayscale_np(arr)
        means.append(float(gray.mean()))

    plt.figure()
    plt.hist(means, bins=30)
    plt.title("Brightness Distribution (0-255)")
    plt.show()

    print(f"Brightness mean: {np.mean(means):.2f}")
    print(f"Brightness std: {np.std(means):.2f}")


In [ ]:
def eda_rgb_distribution(train_dir, n=SAMPLE_N, seed=RANDOM_SEED):
    samples = sample_image_paths(train_dir, n=n, seed=seed)

    r_means, g_means, b_means = [], [], []

    for path, _ in samples:
        arr = pil_to_np(open_rgb(path))
        r_means.append(arr[...,0].mean())
        g_means.append(arr[...,1].mean())
        b_means.append(arr[...,2].mean())

    plt.figure()
    plt.hist(r_means, bins=30, alpha=0.5, label="R")
    plt.hist(g_means, bins=30, alpha=0.5, label="G")
    plt.hist(b_means, bins=30, alpha=0.5, label="B")
    plt.legend()
    plt.title("RGB Distribution")
    plt.show()


In [ ]:
def eda_rgb_distribution(train_dir, n=SAMPLE_N, seed=RANDOM_SEED):
    samples = sample_image_paths(train_dir, n=n, seed=seed)

    r_means, g_means, b_means = [], [], []

    for path, _ in samples:
        arr = pil_to_np(open_rgb(path))
        r_means.append(arr[...,0].mean())
        g_means.append(arr[...,1].mean())
        b_means.append(arr[...,2].mean())

    plt.figure()
    plt.hist(r_means, bins=30, alpha=0.5, label="R")
    plt.hist(g_means, bins=30, alpha=0.5, label="G")
    plt.hist(b_means, bins=30, alpha=0.5, label="B")
    plt.legend()
    plt.title("RGB Distribution")
    plt.show()


In [ ]:
def eda_center_analysis(train_dir, n=SAMPLE_N, seed=RANDOM_SEED):
    samples = sample_image_paths(train_dir, n=n, seed=seed)

    ratios = []

    for path, _ in samples:
        arr = pil_to_np(open_rgb(path))
        gray = to_grayscale_np(arr)

        H, W = gray.shape
        ch, cw = int(H*0.5), int(W*0.5)
        y0, x0 = (H-ch)//2, (W-cw)//2

        full_var = gray.var()
        center_var = gray[y0:y0+ch, x0:x0+cw].var()

        ratios.append(center_var / (full_var + 1e-8))

    plt.figure()
    plt.hist(ratios, bins=30)
    plt.title("Center Variance Ratio")
    plt.show()

    print(f"Center ratio mean: {np.mean(ratios):.3f}")


In [ ]:
def eda_texture_strength(train_dir, n=SAMPLE_N, seed=RANDOM_SEED):
    samples = sample_image_paths(train_dir, n=n, seed=seed)

    strengths = []

    for path, _ in samples:
        arr = pil_to_np(open_rgb(path))
        gray = to_grayscale_np(arr)

        gx = np.abs(gray[:,1:] - gray[:,:-1])
        gy = np.abs(gray[1:,:] - gray[:-1,:])

        strengths.append(gx.mean() + gy.mean())

    plt.figure()
    plt.hist(strengths, bins=30)
    plt.title("Texture / Edge Strength")
    plt.show()

    print(f"Texture mean: {np.mean(strengths):.3f}")


In [ ]:
TRAIN_DIR = './datasets_1000/train'

eda_aspect_ratio_and_size(TRAIN_DIR)
eda_brightness_distribution(TRAIN_DIR)
eda_rgb_distribution(TRAIN_DIR)
eda_center_analysis(TRAIN_DIR)
eda_texture_strength(TRAIN_DIR)


#<font color='pink'> Define image/preprocessing

In [ ]:
# 1. Define global parameters
IMG_SIZE = (200, 200)
BATCH_SIZE = 32
VAL_SPLIT = 0.2
SEED = 42
TRAIN_DIR = './datasets_1000/train'
EPOCHS = 10

print("Global parameters (IMG_SIZE, BATCH_SIZE, VAL_SPLIT, SEED, TRAIN_DIR, EPOCHS) defined.")

ทำอะไร: สร้าง generator แล้วดึง batch 6 รูปมาโชว์เป็น grid 3x2
ทำไม: ตรวจด้วยตาว่า augmentation ที่ตั้งไว้ “สมเหตุสมผล” ไม่บิดภาพจนเพี้ยน

In [ ]:
def visualize_augmented_images(datagen, train_dir, class_mode, img_size):

    train_generator = datagen.flow_from_directory(
        train_dir,
        target_size=img_size,
        batch_size=6,   # 6 รูป
        class_mode=class_mode,
        subset='training',
        seed=SEED
    )

    images, labels = next(train_generator)

    plt.figure(figsize=(10, 8))

    for i in range(6):   #  6 รูป
        plt.subplot(2, 3, i + 1)   # 2 แถว 3 คอลัมน์
        plt.imshow(images[i])
        plt.axis('off')

        if class_mode == 'binary':
            class_name = 'Dog' if labels[i] == 1 else 'Cat'
        else:
            label_map = {v: k for k, v in train_generator.class_indices.items()}
            class_name = label_map[np.argmax(labels[i])]

        plt.title(class_name)

    plt.suptitle("Augmented Images Sample", fontsize=16)
    plt.tight_layout()
    plt.show()

## Refactor Helper Functions


โครง: Conv2D → MaxPool (4 บล็อก) → Flatten → Dense(512) หลายชั้น + BN/Dropout → Dense(1,sigmoid)

ทำไมทำแบบนี้

- Conv หลายชั้นเพื่อเรียนรู้ feature จากง่าย→ยาก

- Dense 512 หลายชั้นเพื่อเพิ่ม capacity

- BatchNorm/Dropout เพื่อช่วยลด overfitting

- Output sigmoid เหมาะกับ binary classification (cat vs dog)

In [ ]:
def build_teacher_cnn():
    model = tf.keras.models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),

        layers.Flatten(),

        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.1),
        layers.BatchNormalization(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.2),
        layers.BatchNormalization(),

        layers.Dense(1, activation='sigmoid')  # binary
    ])
    return model

print("Helper function `build_teacher_cnn` defined.")

ทำอะไร: plot loss/val_loss และ accuracy/val_accuracy

ทำไม: ดู overfitting/underfitting ได้ทันที เช่น train acc ขึ้นแต่ val acc ไม่ขึ้น

In [ ]:
def plot_training_history(history, exp_name):
    history_df = pd.DataFrame(history.history)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1) # First subplot for loss
    history_df.loc[:, ['loss', 'val_loss']].plot(ax=plt.gca())
    plt.title(f'Loss for {exp_name}')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')

    plt.subplot(1, 2, 2) # Second subplot for accuracy
    history_df.loc[:, ['accuracy', 'val_accuracy']].plot(ax=plt.gca())
    plt.title(f'Accuracy for {exp_name}')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')

    plt.tight_layout()
    plt.show()

print("Helper function `plot_training_history` defined.")

ทำอะไร (สำคัญมาก):

- ใช้ datagen.flow_from_directory(... subset="training"/"validation")

- shuffle=False สำหรับ val เพื่อให้ mapping label ตรงตอนทำ confusion matrix

- compile adam + binary_crossentropy + accuracy

- fit และ evaluate

- predict val เพื่อคำนวณ F1-score, print classification report, plot confusion matrix

- return result dict {exp, val_acc, val_loss, f1_score}

ทำไมต้องมี F1 + confusion matrix

- accuracy อย่างเดียวอาจหลอกได้ (โดยเฉพาะถ้าข้อมูลไม่บาลานซ์)

- confusion matrix ทำให้เห็นว่า “สับสน cat→dog” หรือ “dog→cat” ฝั่งไหนหนัก

In [ ]:
def train_and_eval_new(datagen, exp_name, epochs, callbacks=None, class_mode='binary'):
    train_gen = datagen.flow_from_directory(
        TRAIN_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode=class_mode,
        subset="training",
        seed=SEED
    )
    val_gen = datagen.flow_from_directory(
        TRAIN_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode=class_mode,
        subset="validation",
        seed=SEED,
        shuffle=False
    )

    model = build_teacher_cnn()
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    history = model.fit(train_gen, validation_data=val_gen, epochs=epochs, callbacks=callbacks)

    val_loss, val_acc = model.evaluate(val_gen, verbose=0)

    # Get predictions for F1 score
    val_gen.reset() # Reset generator to ensure consistent order
    y_true = val_gen.classes
    y_prob = model.predict(val_gen)
    y_pred = (y_prob > 0.5).astype(int).flatten()

    f1 = f1_score(y_true, y_pred)

    print(f"{exp_name} -> val_acc={val_acc:.4f} | val_loss={val_loss:.4f} | f1_score={f1:.4f}")

    plot_training_history(history, exp_name)

    # Print classification report
    class_labels = sorted(val_gen.class_indices.keys()) # Get class names from class_indices
    print(f"\n--- Classification Report for {exp_name} ---")
    print(classification_report(y_true, y_pred, target_names=class_labels))

    # Plot confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels) # Use class_labels here too
    plt.figure(figsize=(6, 6))
    disp.plot(cmap=plt.cm.Blues, ax=plt.gca())
    plt.title(f'Confusion Matrix for {exp_name}')
    plt.show()

    return model, history, {"exp": exp_name, "val_acc": float(val_acc), "val_loss": float(val_loss), "f1_score": float(f1)}

ทำอะไร: โหลดภาพ 1 รูป → แสดงภาพ → ส่งเข้าโมเดล → print dog/cat


In [ ]:
def predict_single_image(model, image_path, img_size):
    img = load_img(image_path, target_size=img_size)
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Image: {os.path.basename(image_path)}")
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    result = model.predict(img_array)
    if result >= 0.5:
        print("Prediction: Dog")
    else:
        print("Prediction: Cat")
    plt.show()

print("Helper function `predict_single_image` defined.")

## Baseline Model Setup and Training



Baseline ทำอะไร: ไม่มี augmentation ใด ๆ แค่ normalize + split

ทำไมต้องมี baseline: เพื่อใช้เป็น “ฐาน” เทียบว่า augmentation แบบไหนช่วยจริง/ทำแย่ลง

In [ ]:
base_datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=VAL_SPLIT
)

print("Visualizing augmented images for Baseline...")
visualize_augmented_images(base_datagen, TRAIN_DIR, class_mode='binary', img_size=IMG_SIZE)

print("Training Baseline model...")

In [ ]:
baseline_model, history_base, res_baseline = train_and_eval_new(
    base_datagen,
    "Baseline",
    epochs=EPOCHS,
    callbacks=None,
    class_mode='binary'
)

print("Baseline model setup, training, and history plotting complete.")

### Experiment 1: Flip + Rotate

In [ ]:
exp1_datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=VAL_SPLIT,
    rotation_range=20,
    horizontal_flip=True
)

print("Visualizing augmented images for Experiment 1 (Flip + Rotate)...")

In [ ]:
visualize_augmented_images(exp1_datagen, TRAIN_DIR, class_mode='binary', img_size=IMG_SIZE)
print("Training model for Experiment 1 (Flip + Rotate)...")

In [ ]:
from tensorflow.keras.preprocessing import image

model_exp1, hist_exp1, res_exp1 = train_and_eval_new(
    exp1_datagen,
    "Exp1_FlipRotate",
    epochs=EPOCHS,
    class_mode='binary'
)

print("Demonstrating single image prediction for Experiment 1 (Flip + Rotate)...")
predict_single_image(model_exp1, './datasets_1000/train/cats/cat.1.jpg', IMG_SIZE)

### Experiment 2: Zoom + Shift

In [ ]:
exp2_datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=VAL_SPLIT,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1
)

print("Visualizing augmented images for Experiment 2 (Zoom + Shift)...")

In [ ]:
visualize_augmented_images(exp2_datagen, TRAIN_DIR, class_mode='binary', img_size=IMG_SIZE)
print("Training model for Experiment 2 (Zoom + Shift)...")

In [ ]:
model_exp2, hist_exp2, res_exp2 = train_and_eval_new(
    exp2_datagen,
    "Exp2_ZoomShift",
    epochs=EPOCHS,
    class_mode='binary'
)

print("Demonstrating single image prediction for Experiment 2 (Zoom + Shift)...")
predict_single_image(model_exp2, './datasets_1000/train/dogs/dog.1.jpg', IMG_SIZE)

###  Experiment 3: Brightness







In [ ]:
exp3_datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=VAL_SPLIT,
    brightness_range=(0.8, 1.2)
)

print("Visualizing augmented images for Experiment 3 (Brightness)...")

In [ ]:
visualize_augmented_images(exp3_datagen, TRAIN_DIR, class_mode='binary', img_size=IMG_SIZE)
print("Training model for Experiment 3 (Brightness)...")

In [ ]:
model_exp3, hist_exp3, res_exp3 = train_and_eval_new(
    exp3_datagen,
    "Exp3_Brightness",
    epochs=EPOCHS,
    class_mode='binary'
)

print("Demonstrating single image prediction for Experiment 3 (Brightness)...")
predict_single_image(model_exp3, './datasets_1000/train/dogs/dog.17.jpg', IMG_SIZE)

###  Experiment 4: Shear




In [ ]:
exp4_datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=VAL_SPLIT,
    shear_range=0.2
)

print("Visualizing augmented images for Experiment 4 (Shear Transformation)...")

In [ ]:
visualize_augmented_images(exp4_datagen, TRAIN_DIR, class_mode='binary', img_size=IMG_SIZE)
print("Training model for Experiment 4 (Shear Transformation)...")

In [ ]:
model_exp4, hist_exp4, res_exp4 = train_and_eval_new(
    exp4_datagen,
    "Exp4_Shear",
    epochs=EPOCHS,
    class_mode='binary'
)

print("Demonstrating single image prediction for Experiment 4 (Shear Transformation)...")
predict_single_image(model_exp4, './datasets_1000/train/cats/cat.2.jpg', IMG_SIZE)

### Experiment 5: Channel Shift



In [ ]:
exp5_datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=VAL_SPLIT,
    channel_shift_range=0.2
)

print("Visualizing augmented images for Experiment 5 (Channel Shift)...")

In [ ]:
visualize_augmented_images(exp5_datagen, TRAIN_DIR, class_mode='binary', img_size=IMG_SIZE)

print("Training model for Experiment 5 (Channel Shift)...")

In [ ]:
model_exp5, hist_exp5, res_exp5 = train_and_eval_new(
    exp5_datagen,
    "Exp5_ChannelShift",
    epochs=EPOCHS,
    class_mode='binary'
)

print("Demonstrating single image prediction for Experiment 5 (Channel Shift)...")
predict_single_image(model_exp5, './datasets_1000/train/cats/cat.3.jpg', IMG_SIZE)

การปรับโทนสี (Color / Channel Shift)

มันคือการ “ขยับค่าของสี” ในภาพ

เช่น:

- เพิ่มค่า Red ขึ้นเล็กน้อย
- ลดค่า Blue ลง
- ทำให้ภาพอมเหลือง / อมฟ้า

หรือสีดูเพี้ยนจากต้นฉบับเล็กน้อย
- ไม่ได้บิดรูปทรง
- ไม่ได้หมุน
- ไม่ได้ซูม

แต่เปลี่ยน “ค่าสีของพิกเซล”